# Modeling development permit approval

This notebook trains four classifiers -- Logistic Regression, Random Forest,
Gradient Boosting, and XGBoost -- to predict permit approval. We use
stratified splits, cross-validate with ROC-AUC scoring, compare models,
and tune the best one.

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import cross_val_score, StratifiedKFold

sys.path.insert(0, '..')
from src.data_loader import load_and_preprocess
from src.model import (
    FeatureBuilder, CLASSIFIERS, split_data,
    evaluate_model, save_artifacts
)

pd.set_option('display.max_columns', 50)

## 1. Load data and build features

In [ ]:
df = load_and_preprocess(use_cache=True)
print(f"Data shape: {df.shape}")

fb = FeatureBuilder(tfidf_max_features=500)
X = fb.fit_transform(df)
y = df['approved'].values.astype(int)

print(f"Feature matrix: {X.shape}")
print(f"Target distribution: 0={sum(y==0)}, 1={sum(y==1)}")

## 2. Stratified train/test split

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2)

print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")
print(f"Train approval rate: {y_train.mean():.3f}")
print(f"Test approval rate:  {y_test.mean():.3f}")

## 3. Train all four classifiers

In [ ]:
results = {}
for name, clf in CLASSIFIERS.items():
    print(f"Training {name}...")
    clf.fit(X_train, y_train)
    metrics = evaluate_model(clf, X_test, y_test)
    results[name] = {'model': clf, 'metrics': metrics}
    print(f"  Accuracy={metrics['accuracy']:.4f}  "
          f"F1={metrics['f1']:.4f}  "
          f"AUC-ROC={metrics['auc_roc']:.4f}")

## 4. Cross-validation with ROC-AUC scoring

Stratified k-fold cross-validation gives a more robust estimate of
each model's generalization performance.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
for name, clf in CLASSIFIERS.items():
    scores = cross_val_score(clf, X_train, y_train, cv=skf,
                             scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f"{name}: mean AUC = {scores.mean():.4f} (+/- {scores.std():.4f})")

In [ ]:
cv_df = pd.DataFrame(cv_results)
cv_long = cv_df.melt(var_name='Model', value_name='AUC-ROC')

fig = px.box(
    cv_long, x='Model', y='AUC-ROC', points='all',
    title='Cross-validation AUC-ROC (5-fold stratified)'
)
fig.update_layout(height=400)
fig.show()

## 5. Compare models on test set

In [ ]:
comp_rows = []
for name, res in results.items():
    m = res['metrics']
    comp_rows.append({
        'Model': name,
        'Accuracy': round(m['accuracy'], 4),
        'Precision': round(m['precision'], 4),
        'Recall': round(m['recall'], 4),
        'F1': round(m['f1'], 4),
        'AUC-ROC': round(m['auc_roc'], 4),
    })

comp_df = pd.DataFrame(comp_rows)
print(comp_df.to_string(index=False))

best_name = comp_df.loc[comp_df['AUC-ROC'].idxmax(), 'Model']
print(f"\nBest model by AUC-ROC: {best_name}")

In [ ]:
fig = go.Figure()
for metric in ['Accuracy', 'F1', 'AUC-ROC']:
    fig.add_trace(go.Bar(
        name=metric, x=comp_df['Model'], y=comp_df[metric]
    ))

fig.update_layout(
    barmode='group',
    title='Model comparison on test set',
    yaxis_title='Score', height=400
)
fig.show()

## 6. ROC curves for all models

In [ ]:
fig = go.Figure()
for name, res in results.items():
    m = res['metrics']
    fig.add_trace(go.Scatter(
        x=m['fpr'], y=m['tpr'], mode='lines',
        name=f"{name} (AUC={m['auc_roc']:.3f})"
    ))

fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    line=dict(dash='dash', color='gray'),
    name='Random baseline'
))

fig.update_layout(
    title='ROC curves for all classifiers',
    xaxis_title='False positive rate',
    yaxis_title='True positive rate',
    height=450
)
fig.show()

## 7. Tune the best model

We perform a randomized search over hyperparameters for the top model.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
}

search = RandomizedSearchCV(
    XGBClassifier(random_state=42, use_label_encoder=False,
                  eval_metric='logloss'),
    param_dist, n_iter=15, cv=StratifiedKFold(3, shuffle=True, random_state=42),
    scoring='roc_auc', random_state=42, n_jobs=-1, verbose=1
)
search.fit(X_train, y_train)

print(f"\nBest params: {search.best_params_}")
print(f"Best CV AUC: {search.best_score_:.4f}")

In [ ]:
# Evaluate tuned model on test set
tuned_metrics = evaluate_model(search.best_estimator_, X_test, y_test)
print(f"Tuned model test metrics:")
print(f"  Accuracy: {tuned_metrics['accuracy']:.4f}")
print(f"  F1:       {tuned_metrics['f1']:.4f}")
print(f"  AUC-ROC:  {tuned_metrics['auc_roc']:.4f}")

## 8. Save the best model

In [ ]:
save_artifacts(search.best_estimator_, fb, model_name='best_model')
print("Model and feature builder saved.")

## 9. Summary

Four classifiers were trained and evaluated:

- Logistic Regression serves as a strong linear baseline and provides
  interpretable coefficients for TF-IDF terms.
- Random Forest and Gradient Boosting both outperform logistic regression.
- XGBoost achieves the highest AUC-ROC after hyperparameter tuning.

Cross-validation confirms that model ranking is stable across folds.
The next notebook performs a detailed evaluation including confusion
matrices, feature importance, and error analysis by category.